In [1]:
import os
try:
    BASE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE = os.getcwd()
from src.figures import save_figure

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from matplotlib.lines import Line2D

from src.figures import save_figure, DARK, BLUE, RED, GREEN, ORANGE, DIS_COL, GRAY
from src.nn_helpers import load_geo, sigmoid, tanh, relu, GLOBAL_W, LOCAL_W, _L1d, _dL1d, _bowl,_surf3d, _R, _g3, _z_on, _grad
from matplotlib.colors import ListedColormap

warnings.simplefilter(action='ignore', category=FutureWarning)
plt.style.use(os.path.join(BASE,"..", "configs", "visualisations.mplstyle"))

from src.data_loader import _load_disaster_data
df = _load_disaster_data(BASE)

figure_path = os.path.join(BASE, '06_figures')
if not os.path.exists(figure_path):
    os.makedirs(figure_path)

**Fig. 6.1** Motivation

In [2]:
def motivation_panels(df):
    df, _ = load_geo(df=df,types=["Earthquake", "Flood"])
    df = df.sample(min(900, len(df)), random_state=0)
    X = df[["Longitude", "Latitude"]].values.astype(float)
    y = (df["Disaster Type"].values == "Flood").astype(int)
    xx, yy = np.meshgrid(np.linspace(-180, 180, 400), np.linspace(-90, 90, 220))
    grid = np.c_[xx.ravel(), yy.ravel()]
    cmap = ListedColormap(["#E8A9A0", "#A7C3DE"]); A = 0.6

    def sig_feats(P):
        s = 1 / (1 + np.exp(-P / 45.0))
        return np.column_stack([P, s, s ** 2, P[:, 0] * P[:, 1] / 100.0])

    def panel(name, Z, title):
        fig, ax = plt.subplots(figsize=(4.0, 3.5))
        if Z is not None:
            ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5], cmap=cmap, alpha=A)
        for lab, col, nm in [(0, DIS_COL["Earthquake"], "Earthquake"), (1, DIS_COL["Flood"], "Flood")]:
            m = y == lab
            ax.scatter(X[m, 0], X[m, 1], s=10, color=col, alpha=0.9, label=nm, edgecolors="white", linewidth=0.15)
        ax.set_xlim(-180, 180); ax.set_ylim(-90, 90); ax.grid(False)
        ax.set_xlabel("Longitude", fontsize=9); ax.set_ylabel("Latitude", fontsize=9)
        ax.set_title(title, fontsize=12)
        if name == "p0":
            ax.legend(loc="lower left", fontsize=8, markerscale=1.4)
        save_figure(6, 1, 'motivation', variant=name, close=True)

    panel("p0", None, "Raw data")
    lin = LinearRegression().fit(X, y)
    panel("p1", (lin.predict(grid) > 0.5).astype(int).reshape(xx.shape), "Linear regression")
    log = LogisticRegression(max_iter=3000).fit(sig_feats(X), y)
    panel("p2", log.predict(sig_feats(grid)).reshape(xx.shape), "Logistic regression")
    tree = DecisionTreeClassifier(max_depth=5, random_state=0).fit(X, y)
    panel("p3", tree.predict(grid).reshape(xx.shape), "Decision tree (depth 5)")

motivation_panels(df)

Saved Fig. 6.1 -> 06_figures/6.1_motivation/6.1_motivation_p0.pdf
Saved Fig. 6.1 -> 06_figures/6.1_motivation/6.1_motivation_p1.pdf
Saved Fig. 6.1 -> 06_figures/6.1_motivation/6.1_motivation_p2.pdf
Saved Fig. 6.1 -> 06_figures/6.1_motivation/6.1_motivation_p3.pdf


**Fig. 6.2** Activation functions

In [3]:
def fig_activations_plain():
    """Slide 11: activation curves g(z) without derivatives."""
    z = np.linspace(-5, 5, 400)
    funcs = [("Sigmoid", sigmoid(z)), ("Hyperbolic tangent", tanh(z)), ("ReLU", relu(z))]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
    for ax, (name, g) in zip(axes, funcs):
        ax.plot(z, g, color=BLUE, lw=1.8, label="$g(z)$")
        ax.set_title(name, fontsize=11); ax.set_xlabel("$z$")
        ax.axhline(0, color=GRAY, lw=0.5); ax.axvline(0, color=GRAY, lw=0.5)
    save_figure(6, 2, 'activation_functions', close=True)

fig_activations_plain()

Saved Fig. 6.2 -> 06_figures/6.2_activation_functions/6.2_activation_functions.pdf


**Fig. 6.3** Softmax

In [4]:
def softmax_disaster():
    classes = ["Earthquake", "Flood", "Storm", "Volcanic"]
    cols = [DIS_COL["Earthquake"], DIS_COL["Flood"], DIS_COL["Storm"], DIS_COL["Volcanic activity"]]
    logits = np.array([3.1, 0.7, -0.4, 1.2])
    p = np.exp(logits) / np.exp(logits).sum()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
    axes[0].bar(classes, logits, color=GRAY)
    axes[0].axhline(0, color=DARK, lw=0.6)
    axes[0].set_title(r"raw scores (logits)  $Z_m$", fontsize=11)
    axes[0].set_ylabel("score")
    b = axes[1].bar(classes, p, color=cols)
    for i, v in enumerate(p):
        axes[1].text(i, v + 0.015, f"{v:.2f}", ha="center", fontsize=10, weight="bold")
    axes[1].set_ylim(0, 1)
    axes[1].set_title(r"softmax  $f_m(X)$   (sum = %.2f)" % p.sum(), fontsize=11)
    axes[1].set_ylabel("probability")
    axes[1].axhline(1.0, color=DARK, ls=":", lw=1)
    axes[1].text(3.4, 1.0, "1.0", va="bottom", ha="right", fontsize=9, color=DARK)
    for ax in axes:
        ax.tick_params(axis="x", labelsize=9)
    save_figure(6, 3, 'softmax', close=True)

softmax_disaster()

Saved Fig. 6.3 -> 06_figures/6.3_softmax/6.3_softmax.pdf


**Fig. 6.4** Training animation

In [5]:
def nn_over_epochs(df):
    df, types = load_geo(df=df)
    X = df[["Longitude", "Latitude"]].values.astype(float)
    y = df["Disaster Type"].astype(str).values
    scaler = StandardScaler().fit(X)
    Xs = scaler.transform(X)
    clf = MLPClassifier(hidden_layer_sizes=(32, 16), activation="relu",
                        max_iter=1, warm_start=True, random_state=0, learning_rate_init=0.02)
    xx, yy = np.meshgrid(np.linspace(-180, 180, 420), np.linspace(-90, 90, 240))
    grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
    classes = None
    snaps = [1, 3, 8, 20, 60, 150]
    losses = []
    snap_data = {}
    ep = 0
    maxep = max(snaps)
    while ep < maxep:
        clf.fit(Xs, y); ep += 1
        losses.append(clf.loss_)
        if ep in snaps:
            classes = list(clf.classes_)
            code = {c: i for i, c in enumerate(classes)}
            Z = np.vectorize(code.get)(clf.predict(grid)).reshape(xx.shape)
            snap_data[ep] = (Z, list(losses))
    cmap = ListedColormap([DIS_COL[c] for c in classes])

    def draw_map(ax, Z):
        ax.contourf(xx, yy, Z, levels=np.arange(-0.5, len(classes) + 0.5), cmap=cmap, alpha=0.25, zorder=0)
        for c in classes:
            s = df[df["Disaster Type"] == c]
            ax.scatter(s["Longitude"], s["Latitude"], s=5, alpha=0.6, color=DIS_COL[c],
                       edgecolors="white", linewidth=0.15, zorder=2)
        ax.set_xlim(-180, 180); ax.set_ylim(-90, 90)
        ax.set_xlabel("Longitude", fontsize=9); ax.set_ylabel("Latitude", fontsize=9)

    fig, axes = plt.subplots(2, 2, figsize=(11, 6))
    for ax, e in zip(axes.ravel(), [1, 8, 20, 150]):
        draw_map(ax, snap_data[e][0]); ax.set_title(f"epoch {e}", fontsize=11)
    fig.tight_layout()
    save_figure(6, 4, 'training_animation', variant='4up', close=True)

    for i, e in enumerate(snaps):
        Z, lh = snap_data[e]
        fig, (axl, axr) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})
        axl.plot(np.arange(1, len(lh) + 1), lh, color=BLUE, lw=2)
        axl.scatter([len(lh)], [lh[-1]], color=ORANGE, s=30, zorder=5)
        axl.set_xlim(1, maxep); axl.set_ylim(0.6, max(1.25, max(losses[:3]) + 0.05))
        axl.set_xlabel("epoch"); axl.set_ylabel("training loss (cross-entropy)")
        axl.set_title(f"epoch {e}", fontsize=11)
        draw_map(axr, Z)
        save_figure(6, 4, 'training_animation', variant=f'epoch{i}', close=True)

nn_over_epochs(df)

/opt/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1) reached and the optimization hasn't converged yet.
  warnings.warn(
/opt/miniconda3/lib/python3.12/site-packages/sklearn/neural_network/_mul

Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_4up.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch0.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch1.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch2.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch3.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch4.pdf
Saved Fig. 6.4 -> 06_figures/6.4_training_animation/6.4_training_animation_epoch5.pdf


**Fig. 6.5** Gradient descent

In [6]:
def gd_1d_intro():
    """Build-up frames that precede the gradient-descent steps.

    Three frames share the exact axes, limits, title height and labels of
    ``gd_1d_build`` so they overlay pixel-aligned when stacked on one slide:
    (1) empty axes to introduce the axis, (2) the loss curve, (3) the minima.
    """
    ws = np.linspace(-4.6, 4.6, 500)

    def _base():
        fig, ax = plt.subplots(figsize=(8.6, 5.0))
        ax.set_xlabel(r"parameter  $\theta$"); ax.set_ylabel(r"$R(\theta)$")
        ax.set_title("gradient-descent", fontsize=11)  
        ax.set_xlim(-4.6, 4.6); ax.set_ylim(0, 8)
        return fig, ax

    fig, ax = _base()
    save_figure(6, 5, 'gradient_descent', variant='intro_axes', close=True)

    fig, ax = _base()
    ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
    ax.legend(loc="upper center", fontsize=9)
    save_figure(6, 5, 'gradient_descent', variant='intro_curve', close=True)

    fig, ax = _base()
    ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
    ax.scatter([GLOBAL_W, LOCAL_W], [_L1d(GLOBAL_W), _L1d(LOCAL_W)],
               color=GREEN, s=45, zorder=5, label="minima")
    ax.legend(loc="upper center", fontsize=9)
    save_figure(6, 5, 'gradient_descent', variant='intro_minima', close=True)

def gd_1d_build():
    ws = np.linspace(-4.6, 4.6, 500)
    w, eta = -4.2, 0.12
    path = [w]
    for _ in range(7):
        w = w - eta * _dL1d(w); path.append(w)
    for k in range(len(path)):
        fig, ax = plt.subplots(figsize=(8.6, 5.0))
        ax.plot(ws, _L1d(ws), color=BLUE, lw=2.4, label=r"$R(\theta)$")
        ax.scatter([GLOBAL_W, LOCAL_W], [_L1d(GLOBAL_W), _L1d(LOCAL_W)],
                   color=GREEN, s=45, zorder=5, label="minima")
        wk = path[k]; slope = _dL1d(wk); Lk = _L1d(wk)
        tx = np.linspace(wk - 1.3, wk + 1.3, 10)
        ax.plot(tx, Lk + slope * (tx - wk), color=ORANGE, lw=2, ls="--", label=r"$\nabla R$")
        ax.scatter([wk], [Lk], color=RED, s=75, zorder=6)
        if k < len(path) - 1:
            wn = path[k + 1]
            ax.annotate("", xy=(wn, _L1d(wk)), xytext=(wk, _L1d(wk)),
                        arrowprops=dict(arrowstyle="-|>", color=DARK, lw=2))
        ax.set_xlabel(r"parameter  $\theta$"); ax.set_ylabel(r"$R(\theta)$")
        ax.set_title(f"gradient-descent step {k}", fontsize=11)
        ax.legend(loc="upper center", fontsize=9)
        ax.set_xlim(-4.6, 4.6); ax.set_ylim(0, 8)   # FIXED limits so the plot never jumps
        save_figure(6, 5, 'gradient_descent', variant=f'step_{k:02d}', close=True)

gd_1d_intro()
gd_1d_build()

Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_intro_axes.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_intro_curve.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_intro_minima.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_00.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_01.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_02.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_03.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_04.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_05.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_06.pdf
Saved Fig. 6.5 -> 06_figures/6.5_gradient_descent/6.5_gradient_descent_step_07.pdf


**Fig. 6.6** Activation derivatives

In [7]:
def fig_activations():
    z = np.linspace(-5, 5, 400)
    funcs = [("Sigmoid", sigmoid(z), sigmoid(z) * (1 - sigmoid(z))),
             ("Hyperbolic tangent", tanh(z), 1 - tanh(z) ** 2),
             ("ReLU", relu(z), (z >= 0).astype(float))]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
    for ax, (name, g, gp) in zip(axes, funcs):
        ax.plot(z, g, color=BLUE, lw=1.8, label="$g(z)$")
        ax.plot(z, gp, color=ORANGE, lw=1.6, ls="--", label="$g'(z)$")
        ax.set_title(name, fontsize=11)
        ax.set_xlabel("$z$")
        ax.axhline(0, color=GRAY, lw=0.5)
        ax.axvline(0, color=GRAY, lw=0.5)
        ax.legend(loc="upper left", fontsize=9)
    save_figure(6, 6, 'activation_derivatives', close=True)

fig_activations()

Saved Fig. 6.6 -> 06_figures/6.6_activation_derivatives/6.6_activation_derivatives.pdf


**Fig. 6.7** Loss surface

In [ ]:
OUT = os.path.join(figure_path, "loss_surface")
ROT = os.path.join(OUT, "rotation")
DESC = os.path.join(OUT, "descent")

CMAP = "turbo"
ELEV, AZIM = 30, -120        
DESC_AZIM = AZIM + 6 * 14     
ZRATIO = 0.3                
N_ROT = 24                   
START = (-0.6, 1.8)          
LR = 0.8
FIGSIZE = (6.2, 5.2)

_xx = np.linspace(0, np.pi, 4000)
_A = np.array([(2 / np.pi) * np.trapezoid(_xx**3 * (np.pi - _xx) * np.sin(j * _xx), _xx)
               for j in range(1, 4)])
_J = np.arange(1, 4)
_KX, _KY = np.pi / 6.0, np.pi / 3.0
_g = np.linspace(-3, 3, 240)
_X, _Y = np.meshgrid(_g, _g)
_Zraw = sum(_A[k] * np.cos(_J[k] * ((_Y + 3) * _KY)) * np.sin(_J[k] * ((_X + 3) * _KX))
            for k in range(3))
_C = 1.0 / np.max(np.abs(_Zraw))
Z = _Zraw * _C


def loss(x, y):
    """Loss surface value R(x, y)."""
    return _C * np.sum(_A * np.cos(_J * ((y + 3) * _KY)) * np.sin(_J * ((x + 3) * _KX)))


def grad(x, y):
    """Analytic gradient of the loss surface at (x, y)."""
    dfx = _C * np.sum(_A * np.cos(_J * ((y + 3) * _KY)) * np.cos(_J * ((x + 3) * _KX)) * _J * _KX)
    dfy = _C * np.sum(_A * (-np.sin(_J * ((y + 3) * _KY))) * _J * _KY * np.sin(_J * ((x + 3) * _KX)))
    return np.array([dfx, dfy])


_mi = np.unravel_index(np.argmin(Z), Z.shape)
GMIN = (_X[_mi], _Y[_mi], Z[_mi])


def descent_path(start=START, lr=LR, n_steps=11):
    """Return the gradient-descent trajectory as an (n+1, 2) array."""
    p = np.array(start, dtype=float)
    path = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - lr * grad(*p), -3, 3)
        path.append(p.copy())
    return np.array(path)


def _base_axes(fig, azim=AZIM):
    ax = fig.add_subplot(111, projection="3d")
    ax.plot_surface(_X, _Y, Z, cmap=CMAP, rcount=44, ccount=44,
                    linewidth=0.1, edgecolors=(0, 0, 0, 0.18), antialiased=True, alpha=0.95)
    ax.set_box_aspect((1, 1, ZRATIO))
    ax.set_zlim(-1, 1)
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$"); ax.set_zlabel(r"$R(\theta)$")
    for _pax in (ax.xaxis, ax.yaxis, ax.zaxis):     
        _pax.set_pane_color((1, 1, 1, 1))
        _pax.pane.set_edgecolor((0.85, 0.85, 0.85, 1))
    ax.view_init(elev=ELEV, azim=azim)
    return ax


def static_surface():
    fig = plt.figure(figsize=FIGSIZE)
    ax = _base_axes(fig)
    ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=60, zorder=7)
    save_figure(6, 7, 'loss_surface', fig=fig)
    plt.close(fig)
    print("saved loss_surface/loss_surface.pdf")


def rotation_frames(n=N_ROT):
    """One PDF per rotation step; identical fixed canvas for clean overlay."""
    for i in range(n):
        az = AZIM + 360 * i / n
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=az)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=45, zorder=7)
        save_figure(6, 7, 'loss_surface', variant=f'rotation_{i:02d}', fig=fig)
        plt.close(fig)
    print(f"saved rotation/rotation_00..{n - 1:02d}.pdf")

static_surface()
rotation_frames()
print("loss_surface done; global min ~ (%.2f, %.2f)" % (GMIN[0], GMIN[1]))

Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface.pdf
saved loss_surface/loss_surface.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_00.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_01.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_02.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_03.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_04.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_05.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_06.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_07.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_08.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_09.pdf
Saved Fig. 6.7 -> 06_figures/6.7_loss_surface/6.7_loss_surface_rotation_10.pdf
Saved Fig. 6.7 -> 06_figures

**Fig. 6.8 to 6.10** Learning rate regimes

In [9]:
LR_REGIMES = {
    "too_high":    (2.5,  8, "lr_too_large"),
    "too_small":   (0.05, 9, "lr_too_small"),
    "well_chosen": (1.2,  10, "lr_optimal"),
}
START_LR = (2.0, 2.5) 

def lr_descent_frames(regime, lr, number, name, n_steps=30):
    path = descent_path(start=START_LR, lr=lr, n_steps=n_steps)
    zz = np.array([loss(x, y) + 0.03 for x, y in path])
    for k in range(len(path)):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(path[:k + 1, 0], path[:k + 1, 1], zz[:k + 1], "-o",
                color="k", ms=4, lw=1.8, zorder=8)
        ax.scatter([path[0, 0]], [path[0, 1]], [zz[0]], color="magenta", s=55,
                   zorder=9, label=f"start {START_LR}")
        ax.scatter([path[k, 0]], [path[k, 1]], [zz[k]], color="red", s=60, zorder=10)
        ax.set_title(rf"$\rho={lr}$    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:                     
            save_figure(6, number, name, variant=f"step_{k:02d}", fig=fig)
        plt.close(fig)
    print(f"saved Fig. 6.{number} {name}: sampled frames of {len(path)}")

for _regime, (_lr, _num, _name) in LR_REGIMES.items():
    lr_descent_frames(_regime, _lr, _num, _name)

Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_00.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_01.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_02.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_03.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_04.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_05.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_10.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_15.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_20.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_25.pdf
Saved Fig. 6.8 -> 06_figures/6.8_lr_too_large/6.8_lr_too_large_step_30.pdf
saved Fig. 6.8 lr_too_large: sampled frames of 31
Saved Fig. 6.9 -> 06_figures/6.9_lr_too_small/6.9_lr_too_small_step_00.pdf
Saved Fig. 6.9 -> 06_figures/6.9_lr_too_small/6.9_

**Fig. 6.11** SGD mini-batch

In [10]:

def sgd_gradient_noise_frames(theta0=(0.8, 1.2), N=256, seed=1):
    t0 = np.array(theta0, float)
    g_true = grad(*t0)                                  
    rng = np.random.default_rng(seed)
    sigma = 1.3 * np.linalg.norm(g_true)                # per-sample noise level
    samples = g_true + rng.normal(0, sigma, size=(N, 2))  # noisy per-sample gradients
    scale = 1.1 / np.linalg.norm(g_true)                # arrow length scaling

    def arrow(ax, vec, color, lw, alpha=1.0, ls="-", z=5):
        ax.annotate("", xy=t0 - scale * vec, xytext=t0,
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=lw,
                                    alpha=alpha, linestyle=ls), zorder=z)

    for B in [1, 2, 4, 8, 16, 64, N]:
        fig, ax = plt.subplots(figsize=(6.0, 5.6))
        ax.contour(_X, _Y, Z, levels=16, colors="#B9C4CE", linewidths=0.6, zorder=0)
        for s in samples[:B]:                          
            arrow(ax, s, DARK, 1.0, alpha=0.50, z=2)
        arrow(ax, g_true, BLUE, 2.4, ls="--", z=6)      
        arrow(ax, samples[:B].mean(0),  ORANGE, 2.8, z=7)
        ax.scatter(*t0, color="k", s=30, zorder=8)
        name = (f"Gradient decent, {B} sample(s)")
        ax.set_title(name, fontsize=11)
        ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
        ax.set_xlim(-3, 3); ax.set_ylim(-3, 3); ax.set_aspect("equal")
        ax.legend(handles=[
            Line2D([0], [0], color=BLUE, ls="--", lw=2, label="true gradient"),
            Line2D([0], [0], color=ORANGE, lw=2.5, label="batch estimate"),
            Line2D([0], [0], color=DARK, lw=2,alpha=0.50, label="per-sample gradients")],
            loc="upper right", fontsize=8)
        save_figure(6, 11, 'sgd_minibatch', variant=f'batch_{B:03d}', fig=fig)
        plt.close(fig)
    print("saved Fig. 6.11 sgd_minibatch: batch frames B =", [1, 2, 4, 8, 16, 64, N])

sgd_gradient_noise_frames()

Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_001.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_002.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_004.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_008.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_016.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_064.pdf
Saved Fig. 6.11 -> 06_figures/6.11_sgd_minibatch/6.11_sgd_minibatch_batch_256.pdf
saved Fig. 6.11 sgd_minibatch: batch frames B = [1, 2, 4, 8, 16, 64, 256]


**Fig. 6.12** Momentum in 2D

In [11]:
def momentum_frames(start=(2.0, 2.5), rho=0.35, gamma=0.8, n_steps=30):
    p = np.array([start[0] + 0.1, start[1]], float); plain = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - rho * grad(*p), -3, 3); plain.append(p.copy())
    plain = np.array(plain)
    q = np.array(start, float); v = np.zeros(2); mom = [q.copy()]
    for _ in range(n_steps):
        v = gamma * v - rho * grad(*q); q = np.clip(q + v, -3, 3); mom.append(q.copy())
    mom = np.array(mom)
    zp = np.array([loss(x, y) + 0.03 for x, y in plain])
    zm = np.array([loss(x, y) + 0.03 for x, y in mom])
    for k in range(n_steps + 1):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(plain[:k + 1, 0], plain[:k + 1, 1], zp[:k + 1], "-o",
                color="black", ms=3, lw=1.6, zorder=8, label="plain GD")
        ax.plot(mom[:k + 1, 0], mom[:k + 1, 1], zm[:k + 1], "-o",
                color=ORANGE, ms=3.5, lw=2.0, zorder=9, label="momentum")
        ax.scatter([start[0]], [start[1]], [zp[0]], color="magenta", s=50, zorder=10)
        ax.set_title(rf"momentum vs plain GD    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:
            save_figure(6, 12, 'momentum', variant=f'step_{k:02d}', fig=fig)
        plt.close(fig)
    print("saved Fig. 6.12 momentum: sampled frames of", n_steps + 1)

momentum_frames()

Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_00.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_01.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_02.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_03.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_04.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_05.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_10.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_15.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_20.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_25.pdf
Saved Fig. 6.12 -> 06_figures/6.12_momentum/6.12_momentum_step_30.pdf
saved Fig. 6.12 momentum: sampled frames of 31


**Fig. 6.13** Adam in 2D

In [12]:
def adam_frames(start=(2.0, 2.5), rho=0.3, b1=0.9, b2=0.999, eps=1e-8, n_steps=40):
    p = np.array([start[0] + 0.01, start[1]], float); plain = [p.copy()]
    for _ in range(n_steps):
        p = np.clip(p - rho * grad(*p), -3, 3); plain.append(p.copy())
    plain = np.array(plain)
    q = np.array(start, float); m = np.zeros(2); v = np.zeros(2); adm = [q.copy()]
    for t in range(1, n_steps + 1):
        gg = grad(*q)
        m = b1 * m + (1 - b1) * gg
        v = b2 * v + (1 - b2) * gg * gg
        mh = m / (1 - b1 ** t); vh = v / (1 - b2 ** t)
        q = np.clip(q - rho * mh / (np.sqrt(vh) + eps), -3, 3); adm.append(q.copy())
    adm = np.array(adm)
    zp = np.array([loss(x, y) + 0.03 for x, y in plain])
    za = np.array([loss(x, y) + 0.03 for x, y in adm])
    for k in range(n_steps + 1):
        fig = plt.figure(figsize=FIGSIZE)
        ax = _base_axes(fig, azim=DESC_AZIM)
        ax.scatter([GMIN[0]], [GMIN[1]], [GMIN[2]], color="k", marker="*", s=55,
                   zorder=7, label="minimum")
        ax.plot(plain[:k + 1, 0], plain[:k + 1, 1], zp[:k + 1], "-o",
                color="black", ms=3, lw=1.6, zorder=8, label="plain GD")
        ax.plot(adm[:k + 1, 0], adm[:k + 1, 1], za[:k + 1], "-o",
                color=ORANGE, ms=3.5, lw=2.0, zorder=9, label="Adam")
        ax.scatter([start[0]], [start[1]], [za[0]], color="magenta", s=50, zorder=10)
        ax.set_title(rf"Adam vs plain GD    step ${k}$", fontsize=11)
        ax.legend(loc="upper right", fontsize=8)
        if k % 5 == 0 or k < 5:
            save_figure(6, 13, 'adam_2d', variant=f'step_{k:02d}', fig=fig)
        plt.close(fig)
    print("saved Fig. 6.13 Adam: sampled frames of", n_steps + 1)

adam_frames()

Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_00.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_01.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_02.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_03.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_04.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_05.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_10.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_15.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_20.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_25.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_30.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_35.pdf
Saved Fig. 6.13 -> 06_figures/6.13_adam_2d/6.13_adam_2d_step_40.pdf
saved Fig. 6.13 Adam: sampled frames of 41
